In [1]:
import torch

# data set[study hours, sleep hours,Hsc marks]
data = torch.tensor([
    [2,3,4],
    [5,6,4],
    [7,8,5]
])

data

tensor([[2, 3, 4],
        [5, 6, 4],
        [7, 8, 5]])

In [2]:
X = data[:,0:2].float()
y = data[:,2].float()

In [3]:
X

tensor([[2., 3.],
        [5., 6.],
        [7., 8.]])

In [4]:
y

tensor([4., 4., 5.])

Weight Initislization

In [5]:
# way 1
def initialize_parameters():
  parameters = {}

  parameters['W1'] = torch.ones( (2,2) ) * 0.1  # here 0.1 is to balance the weight not too big or low
  parameters['b1'] = torch.zeros( (2,1) )

  parameters['W2'] = torch.ones( (2,1) ) * 0.1
  parameters['b2'] = torch.zeros( (1,1) )

  return parameters

In [6]:
initialize_parameters()

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

Better Way

In [7]:
# way 2
def initialize_parameters(layer_dims):
  torch.manual_seed(3)
  parameters = {}
  L = len(layer_dims)

  for l in range(1, L):
    parameters["W" + str(l)] = torch.ones( (layer_dims[l-1], layer_dims[l]) )*0.1
    parameters["b" + str(l)] = torch.zeros( (layer_dims[l], 1))

  return parameters

In [8]:
initialize_parameters([2,2,1])

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

Forward Propagation

In [9]:
# a = activation
def linear_forward(A_prev, W, b):
  Z = torch.matmul(W.T, A_prev) + b
  return Z

In [10]:
def L_layer_forward(X, parameters):
  A = X                           # A = tensor([[2.],
                                  #             [3.]])
  L = len(parameters)//2          # 4//2 = 2

  for l in range(1, L+1):
    A_prev = A

    wl = parameters['W' + str(l)]   # W1, W2
    bl = parameters['b' + str(l)]   # b1, b2
    A = linear_forward( A_prev, wl, bl)

  return A, A_prev

In [11]:
X

tensor([[2., 3.],
        [5., 6.],
        [7., 8.]])

In [12]:
params = initialize_parameters([2,2,1])
X_sample = X[0].reshape(2,1)
y_hat, A1 = L_layer_forward(X_sample, params)

In [13]:
y_hat

tensor([[0.1000]])

In [14]:
A1

tensor([[0.5000],
        [0.5000]])

In [15]:
print("Error", (y-y_hat)**2)


Error tensor([[15.2100, 15.2100, 24.0100]])


Back Propagation

In [16]:
def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):
  err_signal = 2 * (y-y_hat)

  # Save OLD W2 values before updating
  W2_00_old = parameters['W2'][0,0].item()
  W2_10_old = parameters['W2'][1,0].item()

  # Update Layer 2

  parameters['W2'][0, 0] += lr * err_signal * A1[0, 0]
  parameters['W2'][1, 0] += lr * err_signal * A1[1, 0]
  parameters['b2'][0, 0] += lr * err_signal #

  parameters['W1'][0, 0] += lr * err_signal * parameters['W2'][0, 0] * X[0, 0]
  parameters['W1'][0, 1] += lr * err_signal * parameters['W2'][0, 0] * X[1, 0]
  parameters['b1'][0, 0] += lr * err_signal * parameters['W2'][0, 0]

  parameters['W1'][1, 0] += lr * err_signal * parameters['W2'][1, 0] * X[0, 0]
  parameters['W1'][1, 1] += lr * err_signal * parameters['W2'][1, 0] * X[1, 0]
  parameters['b1'][1, 0] += lr * err_signal * parameters['W2'][1, 0]

  return parameters

In [42]:
# Using Loop

params = initialize_parameters([2,2,1])    # params = {'W1': tensor([[0.1000, 0.1000],
                                           #                        [0.1000, 0.1000]]),
epochs = 15                                #           'b1': tensor([[0.],
                                           #                         [0.]]),
                                           #           'W2': tensor([[0.1000],
                                           #                           [0.1000]]),
for i in range(epochs):                    #           'b2': tensor([[0.]])}
  epoch_loss = 0
  for j in range(X.shape[0]):   #X.shape[0] = row number
    # Prepare sample
      X_sample = X[j].reshape(2, 1)   # input data row X[0].reshape = tensor([[2.],
                                      #                                       [3.]])
      y_sample = y[j]             # y[0] = 4

      # Forward
      y_hat, A1 = L_layer_forward(X_sample, params)  # y_hat, A:tensor([[0.1000]]), A1, A_prev:tensor([[0.5000],[0.5000]])
      y_hat_scalar = y_hat[0, 0]

      # Update
      params = update_parameters(params, y_sample, y_hat_scalar, A1, X_sample)

      # Loss calculation
      loss = (y_sample - y_hat_scalar) ** 2    # MSE
      epoch_loss += loss.item()

  print(f"Epoch - {i+1} Loss - {epoch_loss / X.shape[0]}")

Epoch - 1 Loss - 16.909159660339355
Epoch - 2 Loss - 15.823908170064291
Epoch - 3 Loss - 14.474365870157877
Epoch - 4 Loss - 12.841009457906088
Epoch - 5 Loss - 10.956707318623861
Epoch - 6 Loss - 8.928510983784994
Epoch - 7 Loss - 6.932886441548665
Epoch - 8 Loss - 5.167954603830974
Epoch - 9 Loss - 3.7790237267812095
Epoch - 10 Loss - 2.8071231047312417
Epoch - 11 Loss - 2.196809560060501
Epoch - 12 Loss - 1.8467895438273747
Epoch - 13 Loss - 1.6591690154746175
Epoch - 14 Loss - 1.5626372549061973
Epoch - 15 Loss - 1.5135148875997402
